## **Kahan summation with comparison**

Demonstration: summation of positive random values between 0 and 1

In [1]:
import numpy as np

In [2]:
u = 0.5 * np.finfo(float).eps # Machine epsilon for double precision
u

np.float64(1.1102230246251565e-16)

Sources of error:

- Inherited error
- Algorithm error

For each method, we sum both error sources to obtain a pessimistic upper bound

In [3]:
def sequential(x): # Basic algorithm
    global u

    S = 0.0
    for xi in x:
        S += xi
    
    rel_err = len(x) * u # Total relative error
    err = rel_err * S # Total absolute error

    return S, err, rel_err


def parallel(x): # Parallel addition algorithm
    global u

    a = x.copy() # x copy
    p = (len(a) - 1).bit_length() # ceil(log2(n)) without floating point issues

    while len(a) > 1: # Computing the sum
        b = []

        for i in range(0, len(a) - 1, 2): # Add all complete pairs
            b.append(a[i] + a[i + 1])

        if len(a) % 2: # If odd length, carry the last element unchanged
            b.append(a[-1])

        a = b

    S = a[0] # Sum
    rel_err = u * (p + 1.0) # Total relative error
    err = rel_err * S # Total absolute error

    return S, err, rel_err


def kahan(x): # Kahan summation (basic version, but powerful)
    global u
    
    S = 0.0
    c = 0.0
    for xi in x: # Kahan algorithm
        
        y = xi - c
        t = S + y
        c = (t - S) - y
        S = t
    
    rel_err = 3.0 * u # Total relative error
    err = rel_err * S # Total absolute error
    
    return S, err, rel_err

In [4]:
N = 1_000_000
x = [np.random.random() for _ in range(N)] # One million of random values between 0 and 1

In [5]:
res1, res2, res3, res4 = sequential(x), parallel(x), kahan(x), np.sum(x) # Computing the sum using 4 different methods

In [6]:
res1 # Worst result

(500035.7616634319,
 np.float64(5.5515121573471924e-05),
 np.float64(1.1102230246251565e-10))

In [7]:
res2 # Very good

(500035.7616634499,
 np.float64(1.1658175530429525e-09),
 np.float64(2.3314683517128287e-15))

In [8]:
res3 # Best result. The algorithm error doesn't depend explicity on N

(500035.7616634499,
 np.float64(1.6654536472042178e-10),
 np.float64(3.3306690738754696e-16))

In [9]:
res4 # NumPy uses parallel addition

np.float64(500035.7616634499)